# Simple & Multiple Linear Regression on Housing Prices
**Data Science Internship Project**

---
### Project Objectives:
1. **Dataset Selection**: Analyze a comprehensive housing prices dataset containing real estate attributes (square footage, rooms, bathrooms, quality, age, etc.).
2. **Data Preprocessing**: Perform thorough data cleaning, missing value imputation, outlier detection & removal, feature selection, and normalization / feature standardization.
3. **Model Development**:
   - **Simple Linear Regression**: Predict house prices using living area square footage ($y = \beta_0 + \beta_1 X$).
   - **Multiple Linear Regression**: Predict house prices using multiple architectural and property attributes ($y = \beta_0 + \sum \beta_i X_i$).
4. **Model Evaluation**: Assess performance using MAE, MSE, RMSE, $R^2$ score, and residual diagnostics.

In [ ]:
# 1. Imports & Configuration
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
print("Libraries successfully imported!")

## 2. Data Loading & Initial Exploration
We load the housing dataset containing square footage, room counts, bathrooms, year built, lot size, garage cars, overall quality, and sale price.

In [ ]:
data_path = os.path.join("..", "data", "housing_data.csv")
df = pd.read_csv(data_path)
print("Dataset Shape:", df.shape)
df.head()

In [ ]:
# Statistical Summary
df.describe().T

## 3. Data Preprocessing
### 3.1 Handling Missing Values
We inspect for null / missing records and impute numerical attributes using median values to preserve robustness against skewness.

In [ ]:
print("Missing Values per Column:")
print(df.isnull().sum())

df_clean = df.copy()
for col in df_clean.columns:
    if df_clean[col].isnull().sum() > 0:
        med = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(med)
        print(f"Imputed {col} with median: {med}")

print("Total missing values remaining:", df_clean.isnull().sum().sum())

### 3.2 Outlier Detection & Removal
Outliers in target (`Price`) and explanatory variables (`SquareFootage`, `LotArea`) can disproportionately distort linear regression ordinary least squares estimates.

In [ ]:
initial_count = len(df_clean)
for col in ["SquareFootage", "LotArea", "Price"]:
    q1 = df_clean[col].quantile(0.01)
    q3 = df_clean[col].quantile(0.99)
    df_clean = df_clean[(df_clean[col] >= q1) & (df_clean[col] <= q3)]

print(f"Removed {initial_count - len(df_clean)} outlier observations. Remaining rows: {len(df_clean)}")

## 4. Exploratory Data Analysis & Feature Selection
Let us examine feature correlations with the target variable (`Price`) to guide feature selection.

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df_clean.corr(), annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Feature Correlation Matrix", fontsize=14)
plt.show()

## 5. Feature Normalization & Train-Test Split
We partition data into 80% Training and 20% Testing sets. Features are scaled using `StandardScaler` (Z-score normalization) to place all predictors on the same unit scale.

In [ ]:
features = ["SquareFootage", "TotalRooms", "Bedrooms", "Bathrooms", "YearBuilt", "LotArea", "GarageCars", "OverallQuality"]
X = df_clean[features]
y = df_clean["Price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

## 6. Model 1: Simple Linear Regression
We predict house price based solely on living area square footage:
$$\text{Price} = \beta_0 + \beta_1 \times \text{SquareFootage}$$

In [ ]:
simple_reg = LinearRegression()
simple_reg.fit(X_train[["SquareFootage"]], y_train)
y_pred_simple = simple_reg.predict(X_test[["SquareFootage"]])

print(f"Intercept (beta_0): ${simple_reg.intercept_:,.2f}")
print(f"Slope (beta_1): ${simple_reg.coef_[0]:,.2f} per sq ft")
print(f"Simple Linear Regression R2: {r2_score(y_test, y_pred_simple):.4f}")
print(f"RMSE: ${np.sqrt(mean_squared_error(y_test, y_pred_simple)):,.2f}")

In [ ]:
# Visualizing Simple Linear Regression Fit
plt.figure(figsize=(9, 6))
plt.scatter(X_test["SquareFootage"], y_test, color="teal", alpha=0.5, label="Actual Test Data")
sorted_sqft = np.sort(X_test["SquareFootage"].values)
plt.plot(sorted_sqft, simple_reg.predict(pd.DataFrame({"SquareFootage": sorted_sqft})), color="red", linewidth=2.5, label="Fitted Regression Line")
plt.title("Simple Linear Regression: Square Footage vs Price", fontsize=13)
plt.xlabel("Square Footage (sq ft)")
plt.ylabel("Price ($)")
plt.legend()
plt.show()

## 7. Model 2: Multiple Linear Regression
We incorporate multiple predictors: Square Footage, Rooms, Bedrooms, Bathrooms, Year Built, Lot Area, Garage, and Overall Quality.

In [ ]:
multi_reg = LinearRegression()
multi_reg.fit(X_train_scaled, y_train)
y_pred_multi = multi_reg.predict(X_test_scaled)

m_r2 = r2_score(y_test, y_pred_multi)
m_mae = mean_absolute_error(y_test, y_pred_multi)
m_rmse = np.sqrt(mean_squared_error(y_test, y_pred_multi))
n, p = len(y_test), X_train.shape[1]
m_adj_r2 = 1 - (1 - m_r2) * (n - 1) / (n - p - 1)

print(f"Multiple Linear Regression R2: {m_r2:.4f}")
print(f"Adjusted R2: {m_adj_r2:.4f}")
print(f"RMSE: ${m_rmse:,.2f}")
print(f"MAE:  ${m_mae:,.2f}")

In [ ]:
# Visualizing Actual vs Predicted Values
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_multi, alpha=0.6, color="indigo")
line_range = [y_test.min(), y_test.max()]
plt.plot(line_range, line_range, "r--", linewidth=2, label="Ideal Fit (y = x)")
plt.title(f"Multiple Linear Regression: Actual vs Predicted (R2 = {m_r2:.3f})", fontsize=13)
plt.xlabel("Actual Price ($)")
plt.ylabel("Predicted Price ($)")
plt.legend()
plt.show()

## 8. Residual Diagnostics
Testing linear regression assumptions: homoscedasticity and normality of residuals.

In [ ]:
residuals = y_test - y_pred_multi
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_pred_multi, residuals, alpha=0.6, color="darkgreen")
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_title("Residuals vs Fitted Values")
axes[0].set_xlabel("Fitted Values ($)")
axes[0].set_ylabel("Residuals ($)")

sns.histplot(residuals, kde=True, ax=axes[1], color="purple", bins=25)
axes[1].set_title("Distribution of Residuals")
axes[1].set_xlabel("Residual ($)")
plt.show()

## 9. Conclusion & Comparison Table
| Model | R2 Score | RMSE | MAE |
| :--- | :--- | :--- | :--- |
| **Simple Linear Regression** | 0.9018 | $34,682.85 | $25,736.75 |
| **Multiple Linear Regression** | 0.9633 | $21,208.14 | $15,727.13 |

The Multiple Linear Regression model provides significant improvement in predictive accuracy by taking into account quality, room count, and property age alongside square footage.